✅ 1. 영양기준 모듈 (nutrition_constraint.py)
       질환별 기준 + 복수 질환 병합

→ 2. BMI/허리둘레 → 타겟 열량 계산
       calc_target_energy(bmi, waist_cm) → float
       (영양기준 모듈에 바로 연결되는 입력값)

→ 3. 환자 프로파일 모델
       PatientProfile(질환, BMI, 허리둘레, 식사형태, 예산)
       + 여기서 merge_constraints() 자동 호출

→ 4. Neo4j 메뉴 필터링 쿼리
       질환별 기피/추천 식재료 기반 후보 메뉴 풀 생성
       (Graph-RAG Cypher 쿼리)

→ 5. Simpson 다양성 + NSGA-II 최적화
→ 6. 출력 (식단표, 가이드라인 문서)

In [6]:
from dataclasses import dataclass
from enum import IntEnum
from typing import Optional


class DiseasePriority(IntEnum):
    KIDNEY = 1        # 신장_투석, 신장_비투석 모두 최고 우선순위 (단백질 방향이 반대)
    DIABETES = 2
    CANCER = 2
    HYPERTENSION = 3
    OBESITY = 3


DISEASE_KEY_MAP = {
    "당뇨":       "DIABETES",
    "신장_투석":  "KIDNEY",
    "신장_비투석": "KIDNEY",
    "암":         "CANCER",
    "고혈압":     "HYPERTENSION",
    "비만":       "OBESITY",
    "연하장애":   None,
    "치매":       None,
}


@dataclass
class NutritionConstraint:
    """한 끼 기준 (kcal/영양소)"""
    energy_min:    Optional[float] = None
    energy_max:    Optional[float] = None
    sugar_max:     Optional[float] = None   # g (열량 비례)
    protein_min:   Optional[float] = None
    protein_max:   Optional[float] = None   # 신장_비투석만 상한 존재
    fat_min:       Optional[float] = None
    fat_max:       Optional[float] = None
    sat_fat_max:   Optional[float] = None
    sodium_max:    Optional[float] = None   # mg
    potassium_min: Optional[float] = None   # mg (고혈압만)
    fiber_min:     Optional[float] = None   # g  (고혈압만)


# 모든 질환 공통 열량 범위
ENERGY_MIN = 500  # kcal
ENERGY_MAX = 800  # kcal

DISEASE_CRITERIA = {
    "당뇨": lambda e: NutritionConstraint(
        energy_min=ENERGY_MIN, energy_max=ENERGY_MAX,
        sugar_max=round(e * 0.1 / 4, 2),
        protein_min=18,
        sat_fat_max=round(e * 0.1 / 9, 2),
        sodium_max=1350,
    ),
    "신장_투석": lambda e: NutritionConstraint(
        energy_min=ENERGY_MIN, energy_max=ENERGY_MAX,
        protein_min=round(e * 0.12 / 4, 2),   # 하한 — 충분한 단백질 필요
        sodium_max=650,
    ),
    "신장_비투석": lambda e: NutritionConstraint(
        energy_min=ENERGY_MIN, energy_max=ENERGY_MAX,
        protein_max=round(e * 0.1 / 4, 2),    # 상한 — 단백질 제한 필요
        sodium_max=650,
    ),
    "암": lambda e: NutritionConstraint(
        energy_min=ENERGY_MIN, energy_max=ENERGY_MAX,
        protein_min=round(e * 0.18 / 4, 2),
        fat_min=round(e * 0.15 / 9, 2), fat_max=round(e * 0.35 / 9, 2),
        sat_fat_max=round(e * 0.07 / 9, 2),
        sodium_max=1350,
    ),
    "고혈압": lambda e: NutritionConstraint(
        energy_min=ENERGY_MIN, energy_max=ENERGY_MAX,
        fat_min=round(e * 0.15 / 9, 2), fat_max=round(e * 0.30 / 9, 2),
        sat_fat_max=round(e * 0.07 / 9, 2),
        sodium_max=800,
        potassium_min=700,
        fiber_min=7,
    ),
    "비만": lambda e: NutritionConstraint(
        energy_min=ENERGY_MIN, energy_max=600,  # 비만만 상한 600
        sugar_max=round(e * 0.1 / 4, 2),
        protein_min=18,
        sat_fat_max=round(e * 0.1 / 9, 2),
        sodium_max=1000,
    ),
    "연하장애": lambda e: NutritionConstraint(
        energy_min=ENERGY_MIN, energy_max=ENERGY_MAX,
        # 텍스처 제약만 적용, 영양기준은 기본값
    ),
    "치매": lambda e: NutritionConstraint(
        energy_min=ENERGY_MIN, energy_max=ENERGY_MAX,
        # 텍스처 제약만 적용, 영양기준은 기본값
    ),
}


def merge_constraints(diseases: list[str], energy: float) -> NutritionConstraint:
    """
    우선순위 기반 복수 질환 영양기준 병합.

    단백질 처리 원칙:
      - 신장_투석:   protein_min 확정 (다른 질환 단백질 기준 무시)
      - 신장_비투석: protein_max 확정 (다른 질환 단백질 기준 무시)
      → 신장이 최우선(KIDNEY=1)이므로 암 등 고단백 요구 질환과 충돌 시 신장 기준으로 덮어씀

    나머지 영양소:
      - 나트륨/포화지방/당류: 가장 엄격한(낮은) 값 채택
      - 지방: 교집합 (하한은 최대값, 상한은 최솟값)
      - 열량: 교집합
      - 칼륨/식이섬유: 있으면 적용
    """
    def priority_key(d):
        key = DISEASE_KEY_MAP.get(d)
        return DiseasePriority[key] if key and key in DiseasePriority.__members__ else 99

    sorted_diseases = sorted(diseases, key=priority_key)

    merged = NutritionConstraint()
    kidney_found = any("신장" in d for d in diseases)

    for d in sorted_diseases:
        c = DISEASE_CRITERIA[d](energy)

        # 열량: 교집합
        if c.energy_min: merged.energy_min = max(merged.energy_min or 0,    c.energy_min)
        if c.energy_max: merged.energy_max = min(merged.energy_max or 9999, c.energy_max)

        # 단백질: 신장이 있으면 신장 기준으로 확정, 다른 질환의 단백질 기준 무시
        if "신장" in d:
            merged.protein_min = c.protein_min   # 투석: 하한 설정 / 비투석: None
            merged.protein_max = c.protein_max   # 비투석: 상한 설정 / 투석: None
        elif not kidney_found:
            if c.protein_min: merged.protein_min = max(merged.protein_min or 0, c.protein_min)

        # 나트륨: 가장 엄격한 값
        if c.sodium_max:  merged.sodium_max  = min(merged.sodium_max  or 9999, c.sodium_max)

        # 포화지방: 가장 엄격한 값
        if c.sat_fat_max: merged.sat_fat_max = min(merged.sat_fat_max or 9999, c.sat_fat_max)

        # 당류: 가장 엄격한 값
        if c.sugar_max:   merged.sugar_max   = min(merged.sugar_max   or 9999, c.sugar_max)

        # 지방: 교집합
        if c.fat_min: merged.fat_min = max(merged.fat_min or 0,    c.fat_min)
        if c.fat_max: merged.fat_max = min(merged.fat_max or 9999, c.fat_max)

        # 칼륨/식이섬유: 있으면 적용
        if c.potassium_min: merged.potassium_min = c.potassium_min
        if c.fiber_min:     merged.fiber_min     = c.fiber_min

    return merged

In [8]:
# 복수 질환
c = merge_constraints(["암", "신장_비투석"], energy=650)
print(c)

NutritionConstraint(energy_min=500, energy_max=800, sugar_max=None, protein_min=None, protein_max=16.25, fat_min=10.83, fat_max=25.28, sat_fat_max=5.06, sodium_max=650, potassium_min=None, fiber_min=None)
